# H₂・CO・H–O 電荷交換の設定比較

「H₂ と CO」「H₂ のみ」「分子なし」を、H–O 電荷交換のオン・オフと組み合わせて比較します。各設定で初期の密度・温度・照射条件を揃え、その設定の化学平衡から 10 年、1 万年、100 万年をそれぞれ独立に 1 ステップ更新します。更新時の許容誤差は化学平衡が $10^{-5}$、エネルギー方程式が $10^{-4}$ です。

条件の詳細は [H₂・CO・H–O 電荷交換の設定比較](../README.md#h₂coho-電荷交換の設定比較) を参照してください。計算後は [描画ノートブック](../../plot/PlotMolecularSettings.ipynb) で同じ CSV を読み込めます。

In [1]:
from pathlib import Path

cwd = Path.cwd()
repo_root = next(
    (p for p in (cwd, *cwd.parents) if (p / "test" / "co" / "MolecularSettingsBenchmark.c").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("EQCLib のリポジトリ内から実行してください。")

csv_path = repo_root / "test" / "out" / "co" / "molecular_settings.csv"

## 計算の実行

`rerun = False` にすると、保存済みの CSV を読み込みます。

In [2]:
import subprocess

rerun = True
if rerun:
    completed = subprocess.run(
        ["make", "-C", "test", "molecular-settings-benchmark"],
        cwd=repo_root,
        capture_output=True,
        text=True,
    )
    if completed.returncode:
        print(completed.stdout)
        print(completed.stderr)
        completed.check_returncode()

if not csv_path.is_file():
    raise FileNotFoundError("CSV がありません。rerun = True で計算してください。")
print(csv_path.relative_to(repo_root))

test/out/co/molecular_settings.csv


## 収束状況

各設定で最初の測定ブロックの収束数を表示します。失敗した条件には EQCLib のステータスを表示します。

In [3]:
from IPython.display import Markdown, display
import numpy as np

data = np.genfromtxt(csv_path, names=True, delimiter=",", dtype=None, encoding="utf-8", ndmin=1)
first = data[data["block"] == data["block"].min()]
steps = np.unique(first["dt_years"])
modes = list(dict.fromkeys(first["mode"]))
table = ["| 設定 | " + " | ".join(f"{dt:g} 年" for dt in steps) + " |",
         "|---|" + "---:|" * len(steps)]
for mode in modes:
    counts = []
    for dt in steps:
        rows = first[(first["mode"] == mode) & (first["dt_years"] == dt)]
        counts.append(f"{np.count_nonzero(rows['status'] == 0)} / {len(rows)}")
    table.append(f"| `{mode}` | " + " | ".join(counts) + " |")
display(Markdown("\n".join(table)))

failed = first[first["status"] != 0]
if len(failed):
    table = [
        "| 設定 | Δt [年] | n_H [cm⁻³] | 初期 T [K] | log U | 初期化ステータス | 更新ステータス |",
        "|---|---:|---:|---:|---:|---:|---:|",
    ]
    for r in failed:
        flux = "照射なし" if np.isneginf(r["log_U"]) else f"{r['log_U']:g}"
        table.append(
            f"| `{r['mode']}` | {r['dt_years']:g} | {r['nH']:g} | {r['T0']:g} | {flux} | "
            f"{r['seed_status']} | {r['status']} |"
        )
    display(Markdown("\n".join(table)))
else:
    print("すべての条件が収束しました。")

| 設定 | 10 年 | 10000 年 | 1e+06 年 |
|---|---:|---:|---:|
| `h2_co_ct` | 273 / 273 | 273 / 273 | 272 / 273 |
| `h2_co` | 273 / 273 | 273 / 273 | 272 / 273 |
| `h2_ct` | 273 / 273 | 273 / 273 | 272 / 273 |
| `h2` | 273 / 273 | 273 / 273 | 272 / 273 |
| `atomic_ct` | 273 / 273 | 273 / 273 | 272 / 273 |
| `atomic` | 273 / 273 | 273 / 273 | 272 / 273 |

| 設定 | Δt [年] | n_H [cm⁻³] | 初期 T [K] | log U | 初期化ステータス | 更新ステータス |
|---|---:|---:|---:|---:|---:|---:|
| `h2_co_ct` | 1e+06 | 0.01 | 30 | -5 | 0 | -2 |
| `h2_co` | 1e+06 | 0.01 | 30 | -5 | 0 | -2 |
| `h2_ct` | 1e+06 | 0.01 | 30 | -5 | 0 | -2 |
| `h2` | 1e+06 | 0.01 | 30 | -5 | 0 | -2 |
| `atomic_ct` | 1e+06 | 0.01 | 30 | -5 | 0 | -2 |
| `atomic` | 1e+06 | 0.01 | 30 | -5 | 0 | -2 |